<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c04-header.png" alt="Nextia Learning · SQL and Data Preparation for AI" width="100%">

# Track lineage

**[C04-M06-L02 · Track lineage](https://learning.nextia-ai.com/courses/sql/m06/track-lineage/)** · C04, SQL and Data Preparation for AI · Module 6, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** record where a dataset came from and every step that changed it. You use the extraction manifest and the cleaning counts as lineage records, and you write a dataset card with provenance, licence, synthetic generation, transformation order and versions.

| | |
|---|---|
| **Time** | About 45 minutes |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6. |
| **You should know** | The data contract and `validate.py`, from [Validation rules](https://learning.nextia-ai.com/courses/sql/m06/validation-rules/). |
| **You do not need** | Your local project. The setup below builds the database, the files of Modules 3 to 5 and the validation report here. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every script, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/sql/m06/track-lineage/) has the full explanations, the lineage diagram and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C04/M06-L02-track-lineage/track-lineage-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs `pandas` 3.0.6 in the versions that the course uses. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6
import pandas; print("pandas", pandas.__version__)

## Setup: get the practice data

Run the next cell. It downloads the three files of the Larkfield help-desk export (about 200 KB), checks each file's checksum, and builds the practice database, `larkfield.db`. It works in a folder `ticket-data`, laid out like your project from Module 1, so the code of the lesson runs here unchanged. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: customers 240, tickets 1417, outcomes 1874`. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-data` (in Colab: the **Files** panel on the left), then run the cell again.

This is the same database that `get_data.py` builds in [Set up a practice database](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#build-the-database). The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the Larkfield export, check it, and build larkfield.db in
# ticket-data/. Standard library only: the same steps as get_data.py.
import csv, hashlib, json, os, sqlite3, subprocess, sys, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C04/"
FILES = {
    "customers.csv": "7b437642f0095524f8311f67a7caeb17100d23eed92618b9cf012cd3d3b79993",
    "tickets.csv": "9c47e08cf2ec5977733d32cc21eb97d0dd01b021d1f5fbf7ec31fef6279b1a28",
    "outcomes.csv": "67b4d9d9275b32061a8888050c351df7314054c11ee1ce29e856c1d5403d0653",
}
# The notebook works in its own folder ticket-data/, marked with a file, so it
# never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-data").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, expected):
    path = PROJECT / "data" / "raw" / name
    if not (path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected):
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-data and run again.")
    return path

DB = PROJECT / "larkfield.db"
if not DB.exists():
    paths = {name: fetch(name, sha) for name, sha in FILES.items()}
    build = sqlite3.connect(DB)
    build.executescript("""
        CREATE TABLE customers (customer_id TEXT, segment TEXT, region TEXT, joined_on TEXT);
        CREATE TABLE tickets (ticket_id TEXT, customer_id TEXT, created_at TEXT, channel TEXT, team TEXT,
            priority INTEGER, order_value REAL, word_count INTEGER, first_reply_minutes INTEGER,
            priority_now INTEGER, closed_at TEXT);
        CREATE TABLE outcomes (outcome_id TEXT, ticket_id TEXT, recorded_at TEXT, status TEXT,
            resolution_code TEXT, csat INTEGER);
        CREATE TABLE snapshot (name TEXT, value TEXT);
        INSERT INTO snapshot VALUES ('source', 'Larkfield help-desk export (synthetic, Nextia Learning C04)'),
            ('exported_at', '2026-07-01 06:00:00'), ('data_version', '1.0');
    """)
    for name, path in paths.items():
        with open(path, newline="", encoding="utf-8") as f:
            rows = list(csv.reader(f))
        marks = ", ".join("?" for _ in rows[0])
        build.executemany(f"INSERT INTO {name[:-4]} ({', '.join(rows[0])}) VALUES ({marks})",
                          [[None if v == "" else v for v in row] for row in rows[1:]])
    build.commit()
    build.close()

os.chdir(PROJECT)  # work in the project folder, as in the lesson
for folder in ("queries", "scripts", "reports"):
    Path(folder).mkdir(exist_ok=True)
# Read-only, as q.py: a query cannot change the data by mistake.
con = sqlite3.connect("file:larkfield.db?mode=ro", uri=True)
counts = [con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ("customers", "tickets", "outcomes")]
print("Ready: customers {}, tickets {}, outcomes {}".format(*counts))

## Setup: rebuild the earlier lessons' files

This lesson starts from files that you made in earlier lessons: the extract `data/extract/` from [Reproducible extraction](https://learning.nextia-ai.com/courses/sql/m03/reproducible-extraction/), the clean tables `data/clean/` from [Handle messy values](https://learning.nextia-ai.com/courses/sql/m04/handle-messy-values/), the modelling table `data/model/tickets_model.csv` from [Features and target timing](https://learning.nextia-ai.com/courses/sql/m05/features-and-target-timing/), the train, validation and test sets `data/model/` from [Choose a split](https://learning.nextia-ai.com/courses/sql/m05/choose-a-split/). The next cell downloads the course's finished scripts ([C04/pipeline](https://github.com/usmanahmed99/nextia-learning-labs/tree/main/C04/pipeline)) into `ticket-data` and runs them in order, so you have the same files here. It takes about 20 seconds.

You should see one line per step, ending with `split.py: done`. On your computer, your own scripts from those lessons make the same files.

In [ ]:
# Setup: rebuild the files of the earlier lessons with the course's scripts.
STEPS = ["extract", "clean", "build_features", "split"]
for file in ["queries/extract_tickets.sql", "queries/extract_outcomes.sql"] + [f"scripts/{s}.py" for s in STEPS]:
    if not Path(file).exists():
        download(LABS + "pipeline/" + file, Path(file))
for step in STEPS:
    result = subprocess.run([sys.executable, f"scripts/{step}.py"], capture_output=True, text=True)
    if result.returncode:
        raise SystemExit(f"{step}.py failed:\n{result.stdout}{result.stderr}")
    print(f"{step}.py: done")

## Setup: the helpers

The next cell defines small functions that the lesson uses. You do not need to read them, but you can.

- `sql(query)` runs a query and shows the result as a table, with the number of rows, like `q.py` and the boxes on the lesson page. It shows the first 20 rows; `sql(query, limit=None)` shows all of them.
- `check(query, ...)` compares the result of your query with the expected result, and says what is different. It does not show the answer.
- `expect(what, yours, expected)` compares one of your numbers with the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: show a query result, and check an answer without showing it.
try:
    import pandas as pd
except ImportError:
    pd = None

def run(query, db=None):
    cursor = (db or con).execute(query)
    columns = [d[0] for d in cursor.description] if cursor.description else []
    return columns, cursor.fetchall()

def sql(query, db=None, limit=20):
    """Run a query and show its result as a table, with the row count."""
    columns, rows = run(query, db)
    if not columns:
        print("No result: the cell has no SELECT yet. Write your query between the triple quotes, then run the cell again.")
        return
    shown = rows if limit is None else rows[:limit]
    if pd is not None:
        # Show NULL as NULL, as q.py and the boxes on the lesson page do.
        table = pd.DataFrame(shown, columns=columns, dtype=object)
        display(table.where(table.notna(), "NULL"))
    else:
        print("  ".join(columns))
        for row in shown:
            print("  ".join("NULL" if v is None else str(v) for v in row))
    more = f" (showing {len(shown)}; use limit=None to see all)" if len(shown) < len(rows) else ""
    print(f"{len(rows)} row{'s' if len(rows) != 1 else ''}{more}")

def fingerprint(rows, ordered=False):
    cells = [tuple("NULL" if v is None else f"{v:.6f}" if isinstance(v, float) else str(v) for v in r) for r in rows]
    return hashlib.sha256(repr(cells if ordered else sorted(cells)).encode()).hexdigest()[:16]

def check(query, rows, columns, expected, ordered=False, db=None):
    """Compare your result with the expected one: the column count, the row
    count, then the values (as a fingerprint, so the answer stays hidden)."""
    try:
        got_columns, got = run(query, db)
    except sqlite3.Error as error:
        print(f"Not yet: the query has an error: {error}")
        return
    if len(got_columns) != columns:
        print(f"Not yet: your result has {len(got_columns)} columns; the task asks for {columns}.")
    elif len(got) != rows:
        print(f"Not yet: your result has {len(got)} rows; the task expects {rows}.")
    elif fingerprint(got, ordered) != expected:
        print("Not yet: the number of rows is right, but some values" + (" or their order" if ordered else "") + " are different.")
    else:
        print("Check passed.")

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    same = abs(yours - expected) <= tolerance * max(1, abs(expected)) if isinstance(expected, float) else yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours}, which is not the expected value.")

## Setup: the contract from the last lesson, and a helper

This lesson starts from `scripts/validate.py` and its report `reports/validation.json`, from [Validation rules](https://learning.nextia-ai.com/courses/sql/m06/validation-rules/#the-contract-in-validatepy). The next cell downloads the course's finished `validate.py` from [C04/pipeline](https://github.com/usmanahmed99/nextia-learning-labs/tree/main/C04/pipeline) and runs it.

It also defines `python()`. It runs a script as `python …` does in a terminal, prints the output, and then prints the **exit code** (0 means success).

You should see `All 13 rules passed.` and `[exit code 0]`.

In [ ]:
import json

def python(*args, cwd=None):
    """Run a Python script as `python <args>` in a terminal. Print its output and its exit code."""
    result = subprocess.run([sys.executable, *args], capture_output=True, text=True, cwd=cwd)
    print((result.stdout + result.stderr).rstrip())
    print(f"[exit code {result.returncode}]")
    return result

if not Path("scripts/validate.py").exists():
    download(LABS + "pipeline/scripts/validate.py", Path("scripts/validate.py"))
contract = python("scripts/validate.py")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. One ticket's journey

Priya opens `train.csv` and finds ticket T-10013 with an order value of 32.6. Grace says: "In the help desk, that ticket shows 3260." Who is right? Mei follows the ticket through the files.

> **Predict.** In which file does the value change, and why? Then run the cell.

In [ ]:
sql("SELECT ticket_id, channel, order_value FROM tickets WHERE ticket_id = 'T-10013';")
for path in ["data/extract/tickets.csv", "data/clean/tickets.csv", "data/model/train.csv"]:
    table = pd.read_csv(path, dtype=str, keep_default_na=False, na_values=[""])
    row = table[table["ticket_id"] == "T-10013"].iloc[0]
    print(f"{path:<26} channel {row['channel']:<9} order_value {row['order_value']}")

> **Check.** You should see `webform` and `3260.0` in the database and in the extract, then `web_form` and `32.6` in the clean file and in `train.csv`.

The change happens in `clean.py`: it respelled the channel and divided the cents by 100. The clean file keeps the originals. Run the cell to see them.

In [ ]:
clean = pd.read_csv("data/clean/tickets.csv", dtype=str, keep_default_na=False, na_values=[""])
clean.loc[clean["ticket_id"] == "T-10013", ["ticket_id", "channel", "channel_raw", "order_value", "order_value_raw", "order_value_flag"]]

> **Check.** You should see `channel_raw` = `webform`, `order_value_raw` = `3260.0` and `order_value_flag` = `converted_from_cents`.

Both are right. Grace sees the raw value, in cents from the old web form. Priya sees the corrected value in dollars. Mei can show each step, and the file that records it.

- **Data lineage** is the record of where each piece of data came from, and of every step that changed it, in order.
- **Provenance** is the start of it: the origin of the data. Who made it, when, how, and with what permission.

## 2. The lineage records

Each step of the pipeline writes a small file that records what it did. The [lesson page](https://learning.nextia-ai.com/courses/sql/m06/track-lineage/#the-lineage-of-the-delivered-dataset) shows the whole path as a diagram. Run the cell: it lists the records and checks that each one exists.

In [ ]:
records = {
    "data/extract/manifest.json": "extract.py (Module 3): which rows, which query, how many left out?",
    "reports/cleaning.json": "clean.py (Module 4): how many removed, corrected or quarantined?",
    "data/clean/quarantine.csv": "clean.py (Module 4): which tickets were set aside, and why?",
    "data/model/build_summary.json": "build_features.py (Module 5): how many too recent for a final target?",
    "data/model/split_manifest.json": "split.py (Module 5): which rule made the split, how big is each part?",
    "reports/validation.json": "validate.py (Module 6): did the contract pass?",
}
for path, answer in records.items():
    print(f"{'found  ' if Path(path).exists() else 'MISSING'} {path:<31} {answer}")

> **Check.** You should see `found` six times.

The first record is in the database itself: the `snapshot` table. Run the cell.

In [ ]:
sql("SELECT name, value FROM snapshot;")

> **Check.** You should see 3 rows: `source`, `exported_at` (`2026-07-01 06:00:00`) and `data_version` (`1.0`). Every later record starts from these three values.

`extract.py` copies them into its manifest, with the selection and a checksum of each query. Run the cell.

In [ ]:
extract = json.loads(Path("data/extract/manifest.json").read_text(encoding="utf-8"))
print(json.dumps(extract, indent=2))

> **Check.** You should see the three snapshot values under `source`, the selection `2026-01-01` to `2026-07-01`, `"ticket_rows_not_selected": 1`, and 1416 and 1869 rows. Your `extracted_at` is the time of your run, so it is different from the lesson.

`clean.py` writes the counts of Module 4. They must add up.

> **Predict.** 1,416 rows in, 14 duplicates, 4 quarantined. How many clean rows? Then run the cell.

In [ ]:
cleaning = json.loads(Path("reports/cleaning.json").read_text(encoding="utf-8"))
t = cleaning["tickets"]
print(json.dumps(cleaning, indent=2))
print("in - duplicates - quarantined =", t["input_rows"] - t["exact_duplicates_removed"] - t["quarantined"],
      "| clean_rows =", t["clean_rows"])

> **Check.** The last line is `in - duplicates - quarantined = 1398 | clean_rows = 1398`. Every row is in one place.

## 3. Provenance: where the export came from

**Synthetic data** is data that a program makes, instead of data that a real system records. The Larkfield export comes from `C04/generate.py` in the `nextia-learning-labs` repository. The data is made up for this course; no real people are in it.

**The seed.** The generator uses random numbers. A **seed** is the start value of a random number generator. With the same seed, it makes the same numbers in the same order.

> **Predict.** Do the two lines show the same numbers? And the third line? Then run the cell.

In [ ]:
import random

print("seed 2026:", random.Random(2026).sample(range(100), 5))
print("seed 2026:", random.Random(2026).sample(range(100), 5))
print("seed 2027:", random.Random(2027).sample(range(100), 5))

> **Check.** The first two lines are the same: seed 2026 twice. The third line, with seed 2027, is different. `generate.py` uses seed 2026, so every run writes the same files.

**Checksums.** A **checksum** is a short fingerprint of a file's bytes. SHA-256 is a common method. If one byte changes, the checksum changes completely. The setup cell checked each raw file against `FILES`, the same values as in `get_data.py`. Calculate the checksum of `tickets.csv` yourself. Run the cell.

In [ ]:
import hashlib

mine = hashlib.sha256(Path("data/raw/tickets.csv").read_bytes()).hexdigest()
print("my file:   ", mine)
print("get_data.py", FILES["tickets.csv"])
print("equal:", mine == FILES["tickets.csv"])

> **Check.** You should see `9c47e08cf2ec5977733d32cc21eb97d0dd01b021d1f5fbf7ec31fef6279b1a28` twice and `equal: True`.

**Licence.** A **licence** says what you may do with data. The Larkfield files are under **CC0 1.0**, a public-domain dedication: you may copy, change and share them for any purpose, without permission and without attribution.

> **Warning.** With real data, the licence and the permission come first. Customer tickets can contain personal data. Record the licence and the permission in the provenance before you use the data.

**Versions.** A new export, a changed rule in `clean.py`, or a changed selection gives different rows, so it makes a **new dataset version**. Running the same scripts again on the same snapshot must not change the output. The next lesson proves that with checksums.

## 4. Your turn: copy the numbers from the records

A **dataset card** (also called a datasheet) is a short document that travels with a dataset. Mei copies every number in it from a lineage record, not from memory. First, read the two other records. Run the cell.

In [ ]:
build = json.loads(Path("data/model/build_summary.json").read_text(encoding="utf-8"))
split = json.loads(Path("data/model/split_manifest.json").read_text(encoding="utf-8"))
print(json.dumps(build, indent=2))
print(json.dumps(split, indent=2))

> **Your turn.** Fill in the dictionary `card`. Replace each `None` with an expression that reads the value from a record: `extract`, `cleaning`, `build` or `split`. Do not type the numbers. The first line is done for you. Run the cell, then the check cell.

In [ ]:
card = {
    "selected ticket rows": extract["files"][0]["rows"],   # done for you: from manifest.json
    "ticket rows not selected": None,
    "exact duplicates removed": None,
    "quarantined": None,
    "clean tickets": None,
    "too recent for a final target": None,
    "modelling rows": None,
    "train rows": None,
    "test customers also in train": None,
}
card

In [ ]:
expected_card = {"selected ticket rows": 1416, "ticket rows not selected": 1, "exact duplicates removed": 14,
                 "quarantined": 4, "clean tickets": 1398, "too recent for a final target": 29,
                 "modelling rows": 1369, "train rows": 927, "test customers also in train": 77}
wrong = [key for key, value in expected_card.items() if card.get(key) != value]
print("Check passed." if not wrong else f"Not yet: {len(wrong)} value(s) are not right: {', '.join(wrong)}.")

## 5. Worked example: the dataset card

Mei writes `DATASET.md` in the project folder. The next cell writes her card, the same as on the [lesson page](https://learning.nextia-ai.com/courses/sql/m06/track-lineage/#worked-example-the-dataset-card). Read the sections **Provenance**, **Transformation order** and **Known limitations**. Run the cell. You should see `Writing DATASET.md`.

In [ ]:
%%writefile DATASET.md
# Dataset card: Larkfield ticket escalation dataset

Prepared by Mei (data engineer) for Priya (data scientist). Version 1.0 of the dataset, made from data version 1.0 of the export.

## What it is

One row per support ticket of Larkfield, a fictional online shop for home and garden products. The target, `escalated_72h`, is 1 if a senior agent took the ticket within 72 hours of its creation. Every feature is known when the ticket is created, so a model can use it at that moment.

| File | Rows | What one row is |
|---|---|---|
| `data/model/train.csv` | 927 | a ticket created before 2026-05-01 |
| `data/model/valid.csv` | 227 | a ticket created in May 2026 |
| `data/model/test.csv` | 215 | a ticket created from 2026-06-01 to 2026-06-27 |
| `data/model/tickets_model.csv` | 1,369 | all three parts together |

Escalation rate: 13.7% overall (188 of 1,369); train 11.7%, valid 19.8%, test 16.3%.

## Provenance

| Field | Value |
|---|---|
| Source | The Larkfield help-desk export: `customers.csv`, `tickets.csv`, `outcomes.csv` |
| Snapshot | Exported 2026-07-01 06:00:00, data version 1.0 (the `snapshot` table in `larkfield.db`) |
| How the source was made | **Synthetic.** `C04/generate.py` in `nextia-learning-labs`, Python standard library only, seed 2026. No real system, customer or agent. |
| Licence | CC0 1.0 (public domain). No attribution needed. |
| Raw file checksums (SHA-256) | `customers.csv` 7b437642f0095524f8311f67a7caeb17100d23eed92618b9cf012cd3d3b79993<br>`tickets.csv` 9c47e08cf2ec5977733d32cc21eb97d0dd01b021d1f5fbf7ec31fef6279b1a28<br>`outcomes.csv` 67b4d9d9275b32061a8888050c351df7314054c11ee1ce29e856c1d5403d0653 |
| Selection | Tickets with `2026-01-01 <= created_at < 2026-07-01` |

## Transformation order

| Step | Script | Rows in → out | Lineage record |
|---|---|---|---|
| 0 | `get_data.py` | 3 CSV files → `larkfield.db` (tickets 1,417 rows) | checksums of the raw files |
| 1 | `scripts/extract.py` | 1,417 → 1,416 ticket rows (1 row created in 2036 is outside the selection); 1,874 → 1,869 outcome events | `data/extract/manifest.json` |
| 2 | `scripts/clean.py` | 1,416 → 1,398 tickets (14 exact duplicates removed, 4 quarantined); 1,869 → 1,834 events | `reports/cleaning.json`, `data/clean/quarantine.csv` |
| 3 | `scripts/build_features.py` | 1,398 → 1,369 (29 tickets are too recent for a final target) | `data/model/build_summary.json` |
| 4 | `scripts/split.py` | 1,369 → train 927, valid 227, test 215 | `data/model/split_manifest.json` |
| 5 | `scripts/validate.py` | 13 rules, no change to the data | `reports/validation.json` |

## Columns

| Column | Type | Missing allowed | Meaning |
|---|---|---|---|
| `ticket_id` | text | no | unique, for example `T-10001` |
| `customer_id` | text | yes (guests) | for example `C-0001` |
| `created_at` | datetime | no | `YYYY-MM-DD HH:MM:SS`, 2026-01-05 to 2026-06-27 |
| `channel` | text | no | `email`, `chat`, `phone`, `web_form`, `unknown` (was missing) |
| `team` | text | no | `delivery`, `returns`, `payment`, `warranty`, `account` |
| `priority` | integer | no | 1 (most urgent) to 3, set at creation |
| `segment` | text | no | `home`, `trade`, `business`, `guest` (no customer), `unknown` (deleted customer) |
| `region` | text | no | `north`, `south`, `east`, `west`, `unknown` |
| `order_value` | number | yes | dollars, 0 to 2,000. Missing for account tickets (no order) and where the order system said "unknown" (−1). January web-form values were cents and are divided by 100. |
| `word_count` | integer | no | words in the first message, 0 to 10,000 |
| `customer_tenure_days` | number | yes (guests, deleted customers) | days from `joined_on` to `created_at` |
| `prior_tickets_90d` | integer | no | the same customer's tickets in the 90 days before this one |
| `created_hour` | integer | no | 0 to 23 |
| `escalated_72h` | integer | no | the target: 0 or 1 |

Never in the dataset: `priority_now`, `first_reply_minutes`, `closed_at`, and anything from the outcome events after creation. They are known only after the ticket is created (leakage).

## Tool versions

Tested 2026-10-06 on macOS: Python 3.14.6 (SQLite 3.50.4), pandas 3.0.6.

## Known limitations

- **Synthetic data.** The relations between columns are simple and were chosen for teaching. Do not draw conclusions about real help desks or real customers from this data.
- **The escalation rate changes over time.** It rises after 2026-05-01, when a new warranty policy started (Grace, support lead). Train has 11.7%, valid 19.8%. A model trained on train only sees the old policy.
- **Small test set.** 215 tickets, 35 of them escalated. A difference of a few tickets changes a score noticeably.
- **Customers repeat across parts.** 77 of the 89 customers in test also have tickets in train. Results on test say how the model does for known customers; for new customers they can be worse.
- **Corrected values.** 24 order values were converted from cents. 11 "unknown" values (−1) are now missing. The originals are in `data/clean/tickets.csv` (`order_value_raw`, `order_value_flag`).
- **Excluded rows.** 4 tickets are in quarantine (3 created before the customer joined, 1 with 48,210 words). 29 tickets from the last 72 hours before the snapshot have no final target and are not in the dataset.
- **What not to conclude.** The dataset is for learning data preparation. A model trained on it is a demonstration, not a tool for routing real tickets.

## Changes

| Version | Date | Change |
|---|---|---|
| 1.0 | 2026-10-06 | First delivery, from the export of 2026-07-01 06:00 (data version 1.0). |

Three sections matter most to Priya:

- **Transformation order.** The steps are not interchangeable. `build_features.py` counts `prior_tickets_90d` from the clean tickets, so the 14 duplicates must be gone first.
- **Known limitations.** The facts that a score cannot show: the policy change, the small test set, the customers in train and test (77 of 89, from `split_manifest.json`).
- **What not to conclude.** A dataset card protects the data from wrong use, not only from wrong values.

A card is only right while its numbers match the records. The next cell defines `compare_card()`: it reads the records again and looks for each number in `DATASET.md`. Run the cell.

In [ ]:
def compare_card():
    """Compare the numbers in DATASET.md with the lineage records."""
    extract = json.loads(Path("data/extract/manifest.json").read_text(encoding="utf-8"))
    cleaning = json.loads(Path("reports/cleaning.json").read_text(encoding="utf-8"))["tickets"]
    build = json.loads(Path("data/model/build_summary.json").read_text(encoding="utf-8"))
    split = json.loads(Path("data/model/split_manifest.json").read_text(encoding="utf-8"))
    selection = extract["selection"]
    expected = {
        "selection": f"`{selection['created_from']} <= created_at < {selection['created_before']}`",
        "step 2, clean.py": f"{cleaning['input_rows']:,} → {cleaning['clean_rows']:,} tickets",
        "step 3, build_features.py": f"{build['clean_tickets']:,} → {build['rows']:,}",
        "tickets_model.csv rows": f"| `data/model/tickets_model.csv` | {build['rows']:,} |",
    }
    for name in ["train", "valid", "test"]:
        expected[f"{name}.csv rows"] = f"| `data/model/{name}.csv` | {split['parts'][name]['rows']:,} |"
    text = Path("DATASET.md").read_text(encoding="utf-8")
    different = [what for what, line in expected.items() if line not in text]
    for what, line in expected.items():
        print(f"{'same     ' if what not in different else 'DIFFERENT'}  {what:<26} records: {line}")
    print("The card matches the records." if not different else f"The card does not match the records in {len(different)} place(s).")

compare_card()

> **Check.** You should see `same` seven times and `The card matches the records.`

## 6. Guided practice: trace a ticket

`scripts/trace.py` shows in which files a ticket appears. Run the cell to write it. You should see `Writing scripts/trace.py`.

In [ ]:
%%writefile scripts/trace.py
"""Show where one ticket is in each file of the pipeline."""

import sys
from pathlib import Path

FILES = ["data/extract/tickets.csv", "data/clean/tickets.csv", "data/clean/quarantine.csv",
         "data/model/train.csv", "data/model/valid.csv", "data/model/test.csv"]

ticket = sys.argv[1]
for path in FILES:
    lines = Path(path).read_text(encoding="utf-8").splitlines()
    found = [line for line in lines if line.startswith(ticket + ",")]
    print(f"{path:<26} {len(found)} row(s)")

Run it for T-10013. You should see 1 row in the extract, the clean file and `train.csv`, and 0 rows in the other three files.

In [ ]:
trace = python("scripts/trace.py", "T-10013")

## 7. Your turn: three tickets, three records

> **Predict.** Where are T-10031, T-10361 and T-10147? Write your prediction down. Then run the cell.

In [ ]:
for ticket in ["T-10031", "T-10361", "T-10147"]:
    print(ticket)
    trace = python("scripts/trace.py", ticket)
    print()

> **Your turn.** For each ticket, name the lineage record that explains where it went: `"manifest.json"`, `"cleaning.json"` or `"quarantine.csv"`. Look back at the records in section 2 if you need to. Run the cell, then the check cell.

In [ ]:
# For each ticket, write the record that explains where it went:
# "manifest.json", "cleaning.json" or "quarantine.csv".
explained_by = {
    "T-10031": "",
    "T-10361": "",
    "T-10147": "",
}

In [ ]:
accepted = {"T-10031": {"cleaning.json"}, "T-10361": {"quarantine.csv", "cleaning.json"}, "T-10147": {"manifest.json"}}
wrong = [ticket for ticket, records in accepted.items() if explained_by.get(ticket, "").strip() not in records]
print("Check passed." if not wrong else f"Not yet: look again at {', '.join(wrong)}.")

- **T-10031** is one of the 14 exact duplicates: 2 rows in the extract, 1 after cleaning. The record is `exact_duplicates_removed: 14` in `cleaning.json`.
- **T-10361** is in quarantine, with the reason `word_count_above_10000` in `quarantine.csv`. `cleaning.json` counts it in `quarantined: 4`.
- **T-10147** is in no file. It is the ticket with the year 2036, outside the selection: `ticket_rows_not_selected: 1` in `manifest.json`. A ticket that is in no file still has lineage.

## 8. Failure case: the rules pass, but the card is wrong

Mei runs the extraction again with a different selection, from 2026-02-01, and then the other steps. Nobody changes the card.

> **Predict.** Does `validate.py` pass? Does `compare_card()`? Then run the cell, and the one after it.

In [ ]:
for step in [["scripts/extract.py", "--created-from", "2026-02-01"], ["scripts/clean.py"],
             ["scripts/build_features.py"], ["scripts/split.py"]]:
    print(step[0], "exit code", subprocess.run([sys.executable, *step], capture_output=True).returncode)
contract = python("scripts/validate.py")

In [ ]:
compare_card()

> **Check.** `validate.py` passes all 13 rules. But `compare_card()` finds differences in 5 places: the selection starts at `2026-02-01`, 1,204 → 1,188 tickets, 1,188 → 1,159 rows, the modelling table has 1,159 rows and train has 717. Valid and test do not change: the new selection removes January only.

**Cause:** the new files are valid, so the contract passes. But the contract checks the data, not the document about the data.

**Fix:** compare the card with the records. Decide: is the new selection wanted? If not, run the steps again with the default selection. If yes, it is a new dataset version: update every number in the card from the new records, and add a row to the **Changes** table.

Mei did not want the new selection. Run the cell: it runs the steps again with the default selection and compares again. You should see `The card matches the records.`

In [ ]:
for step in ["scripts/extract.py", "scripts/clean.py", "scripts/build_features.py", "scripts/split.py"]:
    print(step, "exit code", subprocess.run([sys.executable, step], capture_output=True).returncode)
compare_card()

**In AI work:** a model card or a paper that quotes the wrong dataset is very hard to correct later. Nobody knows which data trained the model. The next lesson makes this check automatic: `MANIFEST.json` records the selection and a checksum of every output in each run.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/sql/m06/track-lineage/). They count toward your certificate when you are signed in and enrolled.

In this lesson you followed one ticket through the pipeline and read the lineage records that each step writes. You checked the provenance: a synthetic export, made with seed 2026, under CC0, with checksums. You wrote `DATASET.md` from the records, and you saw that a card that nobody updates is wrong even when every rule passes.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Deliver a reusable dataset](https://learning.nextia-ai.com/courses/sql/m06/deliver-a-reusable-dataset/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/sql/m06/track-lineage/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.